# 서울지역 소방서 
- 서울지역 소방서를 검색하고 csv파일로 저장
- 이후 카카오디벨로퍼 api 위도 경도 받아서 (은평소방서 안됨) 해결방안 찾아서 folium으로 지도에 시각화 표시해보기

- step 1 ( 필요한 모듈 및 파일 업로드)

In [ ]:
!apt-get update > /dev/null 2>&1
!pip install selenium > /dev/null 2>&1
!apt install chromium-chromedriver > /dev/null 2>&1

In [ ]:
import requests
import pandas as pd
import time
import folium
import json
import math
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
from urllib.parse import quote
from tqdm.notebook import tqdm
from bs4 import BeautifulSoup

In [ ]:
options = webdriver.ChromeOptions()
options.add_argument('--headless') # 화면 출력 x
options.add_argument('--no-sandbox') # ?
options.add_argument('--single-process')
options.add_argument('--disable-dev-shm-usage') # /deb/shm 디렉토리를 사용하지 않음 공유메모리를 담당
driver = webdriver.Chrome('chromedriver', options=options)

In [ ]:
from google.colab import files
uploaded = files.upload()
filename = list(uploaded.keys())[0]
filename

Saving roadkey2.txt to roadkey2.txt


'roadkey2.txt'

In [ ]:
with open(filename) as f:
  api_key = f.read()

- step 2 (Selenium을 이용하여 서울소방서 데이터를 추출 후 csv파일로 저장하기)

In [ ]:
url = 'https://www.nfa.go.kr/nfa/introduce/status/firestationidfo/?searchDistance=10&searchMode=distance&myX=37.5605672&myY=126.9433486&searchKeyword='

In [ ]:
driver.get(url)
time.sleep(1)
driver.find_element(By.ID, 'searchKeyword').send_keys('서울')
driver.find_element(By.ID, 'fsSearchBtn').click()

soup = BeautifulSoup(driver.page_source, 'html.parser')
lis = soup.select('.stations-list > li')

strong = soup.select('.stations-local-after > p > strong')
count = int(strong[1].get_text().replace('건', '').strip())
count = math.ceil(count/10)

lines = []
for i in tqdm(range(count)):
    if i >= 1 and i%2 == 0:
        driver.find_element(By.CSS_SELECTOR, '.next_page').click()
        time.sleep(2)
    if i >= 1 and i%2 == 1:
        driver.find_element(By.XPATH, '//*[@id="listForm"]/div/section/ul/li[1]/div/div/ul/li[4]/a').click()
        time.sleep(2)
    soup = BeautifulSoup(driver.page_source, 'html.parser')
    lis = soup.select('.stations-list > li')
    for li in lis:
        name = li.select_one('.title').get_text().strip()
        addr = li.find('address').string.strip()
        tel = li.select_one('.tel').get_text().strip()
        lines.append([name, addr, tel])
df = pd.DataFrame(lines, columns = ['관서명', '주소', '전화번호'])

  0%|          | 0/10 [00:00<?, ?it/s]

In [ ]:
len(lines)

100

In [ ]:
# 은평소방서-수색-119-안전센터 카카오 api검색이 되질않아 삭제
df = df.drop(30, axis=0)
df.head(3)

In [ ]:
df.to_csv('서울소방서.csv', index=False)

- step 3 (추출하여 저장한 csv 파일을 이용하여 위도,경도 데이터 추출)

In [ ]:
fS = pd.read_csv('서울소방서.csv')

In [ ]:
# 하나만 먼저 샘플링
search_url = "https://dapi.kakao.com/v2/local/search/address.json"
sampleTest = quote(fS['주소'][30])
url = f'{search_url}?query={sampleTest}'
result = requests.get(url, headers = {'Authorization': f'KakaoAK {api_key}'}).json()
result

In [ ]:
  testLng = float(result['documents'][0]['address']['x'])
  testLng

126.919788062767

In [ ]:
search_url = "https://dapi.kakao.com/v2/local/search/address.json"
lat_list, lng_list = [], []

for addr in fS['주소']:
  quote(addr)
  url = f'{search_url}?query={quote(addr)}'
  result = requests.get(url, headers = {'Authorization': f'KakaoAK {api_key}'}).json()
  lng = float(result['documents'][0]['address']['x'])
  lat = float(result['documents'][0]['address']['y'])
  lat_list.append(lat)
  lng_list.append(lng)

In [ ]:
fS['위도'] = lat_list
fS['경도'] = lng_list
fS.to_csv('서울소방서(위도경도추가).csv', index = False)

- step 4 (전처리한 csv 데이터로 지도 시각화)

In [ ]:
map = folium.Map(location = [37.541, 126.986], zoom_start = 12, tiles = 'Stamen Terrain')

In [ ]:
fS_df = pd.read_csv('서울소방서(위도경도추가).csv')
fS_df

,관서명,주소,전화번호,위도,경도
0,동작소방서,서울특별시 동작구 여의대방로16길 55(신대방동),02-847-1190,37.494672,126.917719
1,서대문소방서,서울특별시 서대문구 연희로 182(연희동),02-3144-1190,37.573205,126.935996
2,광진소방서,서울특별시 광진구 광나루로 480(구의동),02-457-0119,37.544826,127.082779
3,송파소방서,서울특별시 송파구 오금로51길 56(마천동),02-403-2119,37.499833,127.142592
4,양천소방서,서울특별시 양천구 목동서로 180(목동),02-2655-1119,37.530236,126.872342
...,...,...,...,...,...
94,송파소방서-운동장-119 안전센터,서울특별시 송파구 올림픽로 25 (잠실동),02-2203-2380,37.516200,127.075940
95,송파소방서-잠실-119 안전센터,서울특별시 송파구 석촌호수로 151 (잠실동),02-422-0119,37.507316,127.093893
96,송파소방서-방이-119 안전센터,서울특별시 송파구 강동대로 286 (방이동),02-409-0059,37.519785,127.136572
97,송파소방서-거여-119 안전센터,서울특별시 송파구 마천로 329 (마천동),02-400-0119,37.494163,127.151828


In [ ]:
fS_df

,관서명,주소,전화번호,위도,경도
0,동작소방서,서울특별시 동작구 여의대방로16길 55(신대방동),02-847-1190,37.494672,126.917719
1,서대문소방서,서울특별시 서대문구 연희로 182(연희동),02-3144-1190,37.573205,126.935996
2,광진소방서,서울특별시 광진구 광나루로 480(구의동),02-457-0119,37.544826,127.082779
3,송파소방서,서울특별시 송파구 오금로51길 56(마천동),02-403-2119,37.499833,127.142592
4,양천소방서,서울특별시 양천구 목동서로 180(목동),02-2655-1119,37.530236,126.872342
...,...,...,...,...,...
94,송파소방서-운동장-119 안전센터,서울특별시 송파구 올림픽로 25 (잠실동),02-2203-2380,37.516200,127.075940
95,송파소방서-잠실-119 안전센터,서울특별시 송파구 석촌호수로 151 (잠실동),02-422-0119,37.507316,127.093893
96,송파소방서-방이-119 안전센터,서울특별시 송파구 강동대로 286 (방이동),02-409-0059,37.519785,127.136572
97,송파소방서-거여-119 안전센터,서울특별시 송파구 마천로 329 (마천동),02-400-0119,37.494163,127.151828


In [ ]:
for i in fS_df.index[:]:
  folium.Marker(location = [fS_df.위도[i], fS_df.경도[i]],
                popup = folium.Popup(f'주소. {fS_df.주소[i]} Tel. {fS_df.전화번호[i]}', max_width = 200),
                tooltip = fS_df.관서명[i]).add_to(map)
# title = '<h3 align="center" style="font-size:bold">서울소방서</h3>' 
# map.get_root().html.add_child(folium.Element(title))
map